In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, SiglipProcessor, SiglipModel
import pandas as pd
from tqdm import tqdm
import json
import re
import os
from PIL import Image

# --------------------------------------------------------------------------
# --- 1. 설정: 모델 및 장치 ---
# --------------------------------------------------------------------------
# GPU 장치 설정
LLM_DEVICE = "cuda:0"
SIGLIP_DEVICE = "cuda:0"

# 모델 ID 설정 (Hugging Face Hub ID 직접 사용)
LLM_MODEL_ID = "microsoft/phi-1_5"
SIGLIP_MODEL_ID = "google/siglip-base-patch16-224"

# --------------------------------------------------------------------------
# --- 2. 모델 로드 ---
# --------------------------------------------------------------------------

# (1) LLM 로드 (microsoft/phi-1_5)
print(f"LLM ({LLM_MODEL_ID})을 {LLM_DEVICE}에 로딩합니다...")
try:
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    ).to(LLM_DEVICE)
    llm_model.eval()
    print("✅ LLM 로딩 완료.")
except Exception as e:
    print(f"❌ LLM 모델 로딩 중 오류 발생: {e}")
    exit()

# (2) Vision-Language 모델 로드 (google/siglip-base-patch16-224)
print(f"SigLIP 모델 ({SIGLIP_MODEL_ID})을 {SIGLIP_DEVICE}에 로딩합니다...")
try:
    siglip_processor = SiglipProcessor.from_pretrained(SIGLIP_MODEL_ID)
    siglip_model = SiglipModel.from_pretrained(SIGLIP_MODEL_ID, torch_dtype=torch.bfloat16).to(SIGLIP_DEVICE)
    siglip_model.eval()
    print("✅ SigLIP 모델 로딩 완료.")
except Exception as e:
    print(f"❌ SigLIP 모델 로딩 중 오류 발생: {e}")
    exit()

LLM (microsoft/phi-1_5)을 cuda:0에 로딩합니다...
✅ LLM 로딩 완료.
SigLIP 모델 (google/siglip-base-patch16-224)을 cuda:0에 로딩합니다...
✅ SigLIP 모델 로딩 완료.


In [8]:
# 전체 파라미터와 학습 가능한 파라미터 수를 계산
total_params = sum(p.numel() for p in llm_model.parameters())
trainable_params = sum(p.numel() for p in llm_model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

total_params = sum(p.numel() for p in siglip_model.parameters())
trainable_params = sum(p.numel() for p in siglip_model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}개 ({total_params / 1_000_000:.2f}M)")
print(f"학습 가능한 파라미터: {trainable_params:,}개 ({trainable_params / 1_000_000:.2f}M)")

전체 파라미터: 1,418,270,720개 (1418.27M)
학습 가능한 파라미터: 1,418,270,720개 (1418.27M)
전체 파라미터: 203,155,970개 (203.16M)
학습 가능한 파라미터: 203,155,970개 (203.16M)


In [ ]:
# llm_model
# siglip_model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, SiglipProcessor, SiglipModel
import pandas as pd
from tqdm import tqdm
import json
import re
import os
from PIL import Image

# --------------------------------------------------------------------------
# --- 1. 설정: 모델 및 장치 ---
# --------------------------------------------------------------------------
# GPU 장치 설정
LLM_DEVICE = "cuda:0"
SIGLIP_DEVICE = "cuda:0"

# 모델 ID 설정 (Hugging Face Hub ID 직접 사용)
LLM_MODEL_ID = "microsoft/phi-1_5"
SIGLIP_MODEL_ID = "google/siglip-base-patch16-224"

# --------------------------------------------------------------------------
# --- 2. 모델 로드 ---
# --------------------------------------------------------------------------

# (1) LLM 로드 (microsoft/phi-1_5)
print(f"LLM ({LLM_MODEL_ID})을 {LLM_DEVICE}에 로딩합니다...")
try:
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    ).to(LLM_DEVICE)
    llm_model.eval()
    print("✅ LLM 로딩 완료.")
except Exception as e:
    print(f"❌ LLM 모델 로딩 중 오류 발생: {e}")
    exit()

# (2) Vision-Language 모델 로드 (google/siglip-base-patch16-224)
print(f"SigLIP 모델 ({SIGLIP_MODEL_ID})을 {SIGLIP_DEVICE}에 로딩합니다...")
try:
    siglip_processor = SiglipProcessor.from_pretrained(SIGLIP_MODEL_ID)
    siglip_model = SiglipModel.from_pretrained(SIGLIP_MODEL_ID, torch_dtype=torch.bfloat16).to(SIGLIP_DEVICE)
    siglip_model.eval()
    print("✅ SigLIP 모델 로딩 완료.")
except Exception as e:
    print(f"❌ SigLIP 모델 로딩 중 오류 발생: {e}")
    exit()

# --------------------------------------------------------------------------
# --- 3. 경로 및 프롬프트 정의 ---
# --------------------------------------------------------------------------
base_dir = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/"
input_csv_path = os.path.join(base_dir, "dev_test.csv")
image_base_path = os.path.join(base_dir, "input_images")
output_csv_path = os.path.join("./", "submission.csv")

print(f"입력 파일: {input_csv_path}")
print(f"출력 파일: {output_csv_path}")

# LLM에 사용할 프롬프트 (JSON 출력 유도)
prompt_template = """
You are an expert AI assistant specializing in rephrasing text for visual-language models like CLIP.
Your goal is to convert multiple-choice options into clear, descriptive, and affirmative sentences that can be easily compared to an image.

**Instructions:**
1.  **Create Complete Sentences:** Transform each option from a word or phrase into a full, descriptive sentence.
2.  **Always Be Affirmative:** Create positive, declarative statements.
3.  **Ignore Question Logic:** Completely ignore the framing of the question (e.g., "Which is not..."). Your ONLY job is to rephrase the options (A, B, C, D) into standalone, positive descriptions.
4.  **JSON Output:** Provide your response ONLY as a single JSON object with keys "A", "B", "C", and "D".

### Example
Question: Which of the following foods is NOT present in the image?
A: Pizza
B: Blueberries
C: Salmon
D: Avocado

### Your JSON Output
{{
  "A": "There is a pizza in the photo.",
  "B": "There are blueberries in the photo.",
  "C": "There is a salmon in the photo.",
  "D": "There is an avocado in the photo."
}}

### Your Task
Question: {question}
A: {option_A}
B: {option_B}
C: {option_C}
D: {option_D}

### Your JSON Output
"""

# --------------------------------------------------------------------------
# --- 4. 헬퍼 함수 정의 ---
# --------------------------------------------------------------------------

def rephrase_options_with_llm(row, model, tokenizer, prompt_template, device):
    """LLM으로 보기 4개를 서술형 캡션으로 변환 (파싱 로직 최종 강화)"""
    prompt = prompt_template.format(
        question=row['Question'],
        option_A=row['A'], option_B=row['B'], option_C=row['C'], option_D=row['D']
    )
    
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=512, do_sample=False) # 토큰 길이를 조금 더 여유있게 둠
        
        input_length = inputs.input_ids.shape[1]
        response_content = tokenizer.decode(outputs[0, input_length:], skip_special_tokens=True)
        
        # ⭐️⭐️⭐️ 최종 수정된 JSON 추출 로직 ⭐️⭐️⭐️
        start_index = response_content.find('{')
        if start_index == -1:
            print(f"\nWarning: ID {row.get('ID', 'N/A')} JSON 시작(`{{`)을 찾지 못함. 원본 사용.")
            return [row['A'], row['B'], row['C'], row['D']]

        # 여는 괄호와 닫는 괄호의 개수를 세어 첫 번째 JSON 객체의 끝을 찾음
        brace_count = 0
        end_index = -1
        for i, char in enumerate(response_content[start_index:]):
            if char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
            
            if brace_count == 0:
                end_index = start_index + i + 1
                break
        
        if end_index != -1:
            json_string = response_content[start_index:end_index]
            try:
                processed_options = json.loads(json_string)
                return list(processed_options.values())
            except json.JSONDecodeError as e:
                print(f"\nWarning: ID {row.get('ID', 'N/A')} JSON 파싱 실패 (추출 후). Error: {e}. Raw: '{json_string}'")
                return [row['A'], row['B'], row['C'], row['D']]
        else:
            # JSON 블록이 완결되지 않은 경우
            print(f"\nWarning: ID {row.get('ID', 'N/A')} JSON 블록 완결(`}}`)을 찾지 못함. 원본 사용.")
            return [row['A'], row['B'], row['C'], row['D']]

    except Exception as e:
        print(f"\nError: ID {row.get('ID', 'N/A')} LLM 처리 중 예외: {e}")
        return [row['A'], row['B'], row['C'], row['D']]

def get_similarity_scores(image_path, captions, model, processor, device):
    """SigLIP으로 이미지와 캡션들 간의 유사도 점수 계산"""
    try:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(text=captions, images=image, return_tensors="pt", padding="max_length").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds
        image_embeds = image_embeds / image_embeds.norm(p=2, dim=-1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(p=2, dim=-1, keepdim=True)
        
        similarity_scores = (image_embeds @ text_embeds.T).squeeze(0)
        return similarity_scores.cpu().tolist()

    except Exception as e:
        print(f"\nError: {image_path} SigLIP 처리 중 예외: {e}")
        return [0, 0, 0, 0]

# --------------------------------------------------------------------------
# --- 5. 메인 실행 루프 ---
# --------------------------------------------------------------------------

try:
    df = pd.read_csv(input_csv_path)
    print(f"\n'{input_csv_path}'에서 {len(df)}개 행을 읽었습니다. 추론을 시작합니다.")
except FileNotFoundError:
    print(f"❌ 입력 파일 '{input_csv_path}'을(를) 찾을 수 없습니다.")
    exit()

predictions = []
answer_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

for index, row in tqdm(df.iterrows(), total=len(df), desc="VQA 추론 중"):
    # 1. LLM으로 보기 텍스트를 캡션으로 변환
    rephrased_captions = rephrase_options_with_llm(row, llm_model, llm_tokenizer, prompt_template, LLM_DEVICE)
    
    # 2. SigLIP으로 이미지와 캡션 간 유사도 계산
    image_file_name = os.path.basename(row['img_path'])
    image_path = os.path.join(image_base_path, image_file_name)
    
    scores = get_similarity_scores(image_path, rephrased_captions, siglip_model, siglip_processor, SIGLIP_DEVICE)
    
    # 3. 가장 높은 점수를 받은 보기 선택
    best_option_index = scores.index(max(scores))
    final_answer = answer_map[best_option_index]
    
    predictions.append({'ID': row['ID'], 'answer': final_answer})

# --------------------------------------------------------------------------
# --- 6. 결과 저장 ---
# --------------------------------------------------------------------------

print("\n추론이 완료되었습니다. 제출 파일을 생성합니다.")
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(output_csv_path, index=False)

print(f"\n✅ 모든 작업 완료! 최종 제출 파일이 '{output_csv_path}'에 저장되었습니다.")
print("\n--- 제출 파일 미리보기 (상위 5개) ---")
print(submission_df.head())

LLM (microsoft/phi-1_5)을 cuda:0에 로딩합니다...
✅ LLM 로딩 완료.
SigLIP 모델 (google/siglip-base-patch16-224)을 cuda:0에 로딩합니다...
✅ SigLIP 모델 로딩 완료.
입력 파일: /data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv
출력 파일: ./submission.csv

'/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv'에서 60개 행을 읽었습니다. 추론을 시작합니다.


VQA 추론 중: 100%|██████████| 60/60 [08:17<00:00,  8.29s/it]


추론이 완료되었습니다. 제출 파일을 생성합니다.

✅ 모든 작업 완료! 최종 제출 파일이 './submission.csv'에 저장되었습니다.

--- 제출 파일 미리보기 (상위 5개) ---
         ID answer
0  TEST_000      B
1  TEST_001      B
2  TEST_002      C
3  TEST_003      C
4  TEST_004      B
